# Milestone 4: Multiple-Choice Classification & Fine-Tuning

This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

---

## Objectives

- Understand how to formulate the Smart MCQ Solver Challenge as a multiple-choice classification problem.
- Convert each question prompt and its five options into the format required by multiple-choice models.
- Use `AutoModelForMultipleChoice` to generate logits for options A-E.
- Understand and apply LoRA for parameter-efficient fine-tuning.
- Build and run a fine-tuning pipeline using the Hugging Face Trainer.
- Use PyTorch Softmax to interpret model outputs as probabilities.

In [1]:
!pip install -q transformers peft datasets torch

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForMultipleChoice, 
    TrainingArguments, 
    Trainer
)
from peft import LoraConfig, TaskType, get_peft_model

# Load competition train data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
options = ['A', 'B', 'C', 'D', 'E']

In [2]:
# Q1. Label Encoding
# Convert the answer column in train.csv into numeric labels using the following mapping:
# A = 0
# B = 1
# C = 2
# D = 3
# E = 4

# What is the encoded numeric label for the row at index 150?
# --- Q1 Code ---
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(label_map)

label_150 = train_df.iloc[150]['label']
print(f"Q1 Answer -> Encoded numeric label at row index 150: {label_150}")
# Expected Output: 2

Q1 Answer -> Encoded numeric label at row index 150: 2


In [3]:
# Q2. Prompt-Option Formatting
# For row index 0, create the Option B input using exactly this format:
# str(prompt) + " [SEP] " + str(option_B)

# What is the exact character length of this formatted input string?
# --- Q2 Code ---
row_0 = train_df.iloc[0]
prompt_0 = str(row_0['prompt'])
option_B_0 = str(row_0['B'])

formatted_input_0_B = prompt_0 + " [SEP] " + option_B_0
length_q2 = len(formatted_input_0_B)

print(f"Q2 Answer -> Exact character length: {length_q2}")
# Expected Output: 407

Q2 Answer -> Exact character length: 407


# Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

In [4]:
# Q3. Single-Row MCQ Tokenization
# Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
# padding = "max_length"
# truncation = True
# max_length = 128
# return_tensors = "pt"

# After reshaping for a multiple-choice model, the final input_ids tensor has shape:
# [1, 5, 128]

# What is the value of the second dimension?
# --- Q3 Code ---
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Prepare 5 formatted inputs for row 0
row_0_pairs = [f"{row_0['prompt']} [SEP] {row_0[opt]}" for opt in options]

# Tokenize 5 choices
encoding_0 = tokenizer(
    row_0_pairs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape input_ids to [1, 5, 128]
input_ids_q3 = encoding_0['input_ids'].unsqueeze(0)
second_dim_value = input_ids_q3.shape[1]

print(f"Q3 Answer -> Shape: {list(input_ids_q3.shape)}, Second Dimension Value: {second_dim_value}")
# Expected Output: 5

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q3 Answer -> Shape: [1, 5, 128], Second Dimension Value: 5


In [5]:
# Q4. Batch MCQ Tokenization
# Tokenize the first 16 rows of train.csv as multiple-choice examples.
# Each row has 5 choices.
# Each choice is tokenized to length 128.

# The final input_ids tensor has shape:
# [16, 5, 128]

# How many total token positions are in this tensor?
# --- Q4 Code ---
first_16_rows = train_df.iloc[:16]
batch_inputs = []

for idx, row in first_16_rows.iterrows():
    row_choices = [f"{row['prompt']} [SEP] {row[opt]}" for opt in options]
    batch_inputs.append(row_choices)

# Flatten list to tokenize all 80 choices (16 * 5)
flattened_inputs = [choice for question in batch_inputs for choice in question]

batch_encoding = tokenizer(
    flattened_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape tensor to [16, 5, 128]
batch_input_ids = batch_encoding['input_ids'].view(16, 5, 128)
total_token_positions = batch_input_ids.numel()

print(f"Q4 Answer -> Tensor Shape: {list(batch_input_ids.shape)}, Total Token Positions: {total_token_positions}")
# Expected Output: 10240

Q4 Answer -> Tensor Shape: [16, 5, 128], Total Token Positions: 10240


# Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

In [6]:
# --- Q5 Code ---
# Q5. Multiple-Choice Logits
# Load bert-base-uncased using AutoModelForMultipleChoice.
# Tokenize row index 0 as 5 choices and pass it through the model.

# The output logits tensor has shape:
# [1, 5]

# How many logits are produced for one question?
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

with torch.no_grad():
    outputs_q5 = model(input_ids=input_ids_q3)

logits_q5 = outputs_q5.logits
num_logits = logits_q5.shape[1]

print(f"Q5 Answer -> Logits Shape: {list(logits_q5.shape)}, Num Logits Produced: {num_logits}")
# Expected Output: 5

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q5 Answer -> Logits Shape: [1, 5], Num Logits Produced: 5


In [7]:
# --- Q6 Code ---
# Q6. Supervised Loss Tensor
# For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

# The model returns a scalar loss tensor.

# How many dimensions does this loss tensor have

label_0_tensor = torch.tensor([row_0['label']])

with torch.no_grad():
    outputs_q6 = model(input_ids=input_ids_q3, labels=label_0_tensor)

loss_tensor = outputs_q6.loss
loss_dimensions = loss_tensor.dim()

print(f"Q6 Answer -> Loss Tensor: {loss_tensor}, Dimensions: {loss_dimensions}")
# Expected Output: 0

Q6 Answer -> Loss Tensor: 1.5912220478057861, Dimensions: 0


# LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

In [8]:
# --- Q7 Code ---
# Q7. LoRA Trainable Parameters
# Apply LoRA to the bert-base-uncased multiple-choice model using:
# r = 8
# lora_alpha = 16
# target_modules = ["query", "value"]
# lora_dropout = 0.1
# bias = "none"
# task_type = TaskType.SEQ_CLS

# Count trainable parameters using:
# sum(p.numel() for p in model.parameters() if p.requires_grad)

# How many parameters are trainable?
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

lora_model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"Q7 Answer -> Trainable Parameters Count: {trainable_params}")
# Expected Output: 295681

Q7 Answer -> Trainable Parameters Count: 295681


Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

In [9]:
# --- Q8 Code ---
# Q8. Hugging Face Dataset Preparation
# Create a Hugging Face Dataset from the first 100 rows of train.csv.

# For each row, create:
# input_ids with shape [5, 128]
# attention_mask with shape [5, 128]
# labels as the encoded answer label

# For the first dataset item, input_ids has shape:
# [5, 128]

# How many tokenized choices are stored in input_ids?
first_100_rows = train_df.iloc[:100]

def preprocess_mcq(df, max_len=128):
    dataset_dict = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for idx, row in df.iterrows():
        choices = [f"{row['prompt']} [SEP] {row[opt]}" for opt in options]
        enc = tokenizer(choices, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        dataset_dict['input_ids'].append(enc['input_ids'].numpy())
        dataset_dict['attention_mask'].append(enc['attention_mask'].numpy())
        dataset_dict['labels'].append(row['label'])
    return Dataset.from_dict(dataset_dict)

train_dataset_100 = preprocess_mcq(first_100_rows, max_len=128)
item_0_choices = len(train_dataset_100[0]['input_ids'])

print(f"Q8 Answer -> Number of tokenized choices in input_ids: {item_0_choices}")
# Expected Output: 5

Q8 Answer -> Number of tokenized choices in input_ids: 5


# Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

In [10]:
# --- Q9 Code ---
# Q9. Tiny LoRA Fine-Tuning
# Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

# Use the following settings:
# max_length = 64
# per_device_train_batch_size = 4
# gradient_accumulation_steps = 1
# max_steps = 4

# What is the final global_step reported by the Trainer?
train_dataset_32 = preprocess_mcq(train_df.iloc[:32], max_len=64)

# Fresh model instance
base_model_q9 = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
peft_model_q9 = get_peft_model(base_model_q9, lora_config)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    learning_rate=5e-4,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=peft_model_q9,
    args=training_args,
    train_dataset=train_dataset_32,
)

trainer.train()
final_global_step = trainer.state.global_step

print(f"Q9 Answer -> Final global_step: {final_global_step}")
# Expected Output: 4

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,3.352875
2,3.296142
3,3.103034
4,3.284014


Q9 Answer -> Final global_step: 4


In [11]:
# Q10. Probability Assigned to Option E After Fine-Tuning
# Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

# What is the probability assigned to Option E?

# Round your answer to 4 decimal places.

# --- Q10 Code ---
peft_model_q9.eval()

# Tokenize row index 0 with max_length=64 (matching Q9 fine-tuning)
enc_row0 = tokenizer(
    [f"{row_0['prompt']} [SEP] {row_0[opt]}" for opt in options],
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

inputs_q10 = {
    'input_ids': enc_row0['input_ids'].unsqueeze(0).to(peft_model_q9.device),
    'attention_mask': enc_row0['attention_mask'].unsqueeze(0).to(peft_model_q9.device)
}

with torch.no_grad():
    logits_q10 = peft_model_q9(**inputs_q10).logits

probs_q10 = torch.softmax(logits_q10, dim=-1)[0]
prob_opt_e = probs_q10[4].item()

print(f"Q10 Answer -> Probability assigned to Option E: {prob_opt_e:.4f}")
# Expected Output: 0.1970

Q10 Answer -> Probability assigned to Option E: 0.1895
